# K-means en 2D: evaluación con etiquetas verdaderas

Este notebook mantiene el caso de dos distribuciones gaussianas separadas, pero ahora conserva las etiquetas verdaderas usadas durante la simulación. Esto permite comparar el agrupamiento producido por K-means contra el origen real de los datos.

Aunque K-means es un método no supervisado, en ejemplos simulados sí podemos conocer la clase real de cada punto y usarla para evaluar el resultado.

## 1. Precaución importante sobre las etiquetas de K-means

K-means no sabe qué significa “clase 0” o “clase 1”. El algoritmo solo produce identificadores de clusters.

Por ejemplo, el grupo generado como clase real 0 podría ser nombrado por K-means como cluster 1. Esto no es un error: simplemente los nombres de los clusters son arbitrarios.

Por esta razón, antes de calcular el acierto se debe buscar la mejor correspondencia entre:

- etiquetas verdaderas;
- etiquetas asignadas por K-means.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from itertools import permutations
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, adjusted_rand_score, confusion_matrix

RANDOM_STATE = 7
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams["figure.figsize"] = (7, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

In [ ]:
def mejor_acierto_por_permutacion(y_true, y_cluster):
    """Calcula el mejor acierto posible cuando los nombres de los clusters son arbitrarios.

    K-means no sabe que una clase debe llamarse 0, 1, 2, etc. Por eso, antes de
    calcular accuracy, se prueban todas las correspondencias posibles entre
    clusters encontrados y etiquetas verdaderas.
    """
    y_true = np.asarray(y_true)
    y_cluster = np.asarray(y_cluster)

    etiquetas_verdaderas = np.unique(y_true)
    etiquetas_cluster = np.unique(y_cluster)

    if len(etiquetas_verdaderas) != len(etiquetas_cluster):
        raise ValueError("El número de etiquetas verdaderas y clusters debe coincidir.")

    mejor_accuracy = -1.0
    mejor_mapeo = None
    mejor_y_pred = None

    for perm in permutations(etiquetas_verdaderas):
        mapeo = {cluster: etiqueta for cluster, etiqueta in zip(etiquetas_cluster, perm)}
        y_pred = np.array([mapeo[c] for c in y_cluster])
        acc = accuracy_score(y_true, y_pred)

        if acc > mejor_accuracy:
            mejor_accuracy = acc
            mejor_mapeo = mapeo
            mejor_y_pred = y_pred

    return mejor_accuracy, mejor_y_pred, mejor_mapeo

## 2. Generación de datos y etiquetas verdaderas

Se generan dos nubes de puntos. Además, se crea un vector `true_labels`, donde:

- `0` identifica los puntos generados desde la primera distribución;
- `1` identifica los puntos generados desde la segunda distribución.

In [ ]:
mu1 = np.array([2, 3])
sigma1 = np.array([[1, 0.5],
                   [0.5, 1]])

mu2 = np.array([6, 8])
sigma2 = np.array([[1, -0.5],
                   [-0.5, 1.5]])

n_points = 100

data1 = rng.multivariate_normal(mu1, sigma1, n_points)
data2 = rng.multivariate_normal(mu2, sigma2, n_points)

data = np.vstack([data1, data2])
true_labels = np.r_[np.zeros(n_points, dtype=int), np.ones(n_points, dtype=int)]

df = pd.DataFrame(data, columns=["dimension_1", "dimension_2"])
df["etiqueta_verdadera"] = true_labels
df.head()

## 3. Visualización usando las etiquetas verdaderas

Esta gráfica se usa solo para fines didácticos. En un problema real de clustering, normalmente no se dispone de estas etiquetas.

In [ ]:
plt.figure()
plt.scatter(data[:, 0], data[:, 1], c=true_labels, cmap="tab10", s=40, alpha=0.85)
plt.title("Datos 2D según las etiquetas verdaderas")
plt.xlabel("Dimensión 1")
plt.ylabel("Dimensión 2")
plt.show()

## 4. Aplicación de K-means

Se usa `KMeans` de `scikit-learn` con dos clusters.

In [ ]:
num_clusters = 2

kmeans = KMeans(n_clusters=num_clusters, random_state=RANDOM_STATE, n_init=10)
cluster_idx = kmeans.fit_predict(data)
cluster_centers = kmeans.cluster_centers_

pd.DataFrame(cluster_centers, columns=["centroide_x", "centroide_y"])

## 5. Visualización de los clusters encontrados

La siguiente figura muestra el resultado de K-means. Los centroides aparecen como estrellas negras.

In [ ]:
plt.figure()
plt.scatter(data[:, 0], data[:, 1], c=cluster_idx, cmap="tab10", s=40, alpha=0.85)
plt.scatter(cluster_centers[:, 0], cluster_centers[:, 1], marker="*", s=300, c="black", edgecolor="white", label="Centroides")
plt.title("Clusters asignados por K-means")
plt.xlabel("Dimensión 1")
plt.ylabel("Dimensión 2")
plt.legend()
plt.show()

## 6. Cálculo del acierto

No se calcula el acierto directamente con `cluster_idx`, porque las etiquetas de K-means pueden estar intercambiadas.

Primero se encuentra la mejor correspondencia entre clusters y etiquetas verdaderas. Luego se calcula:

$$
\text{accuracy} = \frac{\text{muestras correctamente agrupadas}}{\text{total de muestras}}
$$

También se calcula el **Adjusted Rand Index**. Esta métrica es útil en clustering porque compara particiones sin depender del nombre asignado a cada cluster.

In [ ]:
accuracy, predicted_labels, mapping = mejor_acierto_por_permutacion(true_labels, cluster_idx)
ari = adjusted_rand_score(true_labels, cluster_idx)

print(f"Mejor correspondencia cluster → etiqueta verdadera: {mapping}")
print(f"Número de muestras correctamente agrupadas: {int(accuracy * len(true_labels))} de {len(true_labels)}")
print(f"Accuracy corregido por permutación: {accuracy:.4f}")
print(f"Adjusted Rand Index: {ari:.4f}")

## 7. Matriz de confusión después de corregir las etiquetas

Luego de mapear los clusters a las etiquetas verdaderas, la matriz de confusión permite contar errores y aciertos por clase.

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)
cm_df = pd.DataFrame(cm,
                     index=["Clase real 0", "Clase real 1"],
                     columns=["Predicha 0", "Predicha 1"])
cm_df

## 8. Actividad propuesta

Acerque las distribuciones modificando `mu1` y `mu2`, y observe cómo cambian:

1. el número de muestras correctamente agrupadas;
2. la matriz de confusión;
3. el Adjusted Rand Index.

La pregunta central es: ¿en qué momento la separación visual deja de ser suficiente para que K-means recupere correctamente los dos grupos?